# 09. Real Colab Runtime Validation & Storage Benchmark

End-to-end executable notebook for Google Colab validating environment, pure-C Colibrì v1.5.0+ build, Google Drive mounting (`AI - Google Drive` ID: `11BdZx7pI2XyEmiJjpZJjTCIX1V41vKhd`, account: `aqibjawwad2607@gmail.com`), 142-shard model integrity verification, live inference execution, and comparative storage benchmarking.

## Step 1: Fresh Colab Hardware & Environment Diagnostic

In [ ]:
!pip install -q pyyaml psutil rich tabulate requests fastapi uvicorn httpx pydantic

# Run environment capability check
!python scripts/environment_check.py

## Step 2: Toolchain Setup & Colibrì Pure-C Build

In [ ]:
# Install Linux build essentials & OpenMP runtime
!apt-get update -qq && apt-get install -y -qq build-essential libgomp1 git

# Clone Colibri source
!rm -rf /content/colibri
!git clone https://github.com/JustVugg/colibri.git /content/colibri

# Build optimized GLM engine
%cd /content/colibri
!make glm ARCH=native
!./coli doctor
%cd /content

## Step 3: Google Drive Mount & Model Directory Verification

In [ ]:
import os
import importlib

# Mount Google Drive (Authenticate with account: aqibjawwad2607@gmail.com)
try:
    drive = importlib.import_module('google' + '.colab.drive')
    drive.mount('/content/drive', force_remount=False)
except Exception as e:
    print(f"Note: Drive mount skipped in local development ({e})")

DRIVE_MODEL_DIR = '/content/drive/MyDrive/AI - Google Drive/GLM-5.2/model'
print(f"Target model path in Drive: {DRIVE_MODEL_DIR}")

# Verify storage quota, folder ID structure, and read/write health
!python scripts/drive_check.py --path "{DRIVE_MODEL_DIR}" --required-gb 400

## Step 4: 142-Shard Safetensors Header & MTP Verification

In [ ]:
# Run inventory and non-destructive header verification
!python scripts/model_inventory.py --model-dir "{DRIVE_MODEL_DIR}"
!python scripts/model_verify.py --model-dir "{DRIVE_MODEL_DIR}" --expected-shards 142

## Step 5: Runtime Configuration & Real Colibrì Inference

In [ ]:
# Set runtime environment flags
os.environ['COLI_MODEL'] = DRIVE_MODEL_DIR
os.environ['COLI_RAM'] = '16'
os.environ['COLI_CAP'] = '256'
os.environ['COLI_REPIN'] = '1'

# Execute deterministic validation test prompts
!/content/colibri/coli chat --model "{DRIVE_MODEL_DIR}" --prompt "Hello. Respond with exactly one sentence."

## Step 6: Empirical Storage Benchmark (Local NVMe vs Google Drive FUSE)

In [ ]:
LOCAL_MODEL_DIR = '/content/model'
BENCHMARK_OUTPUT = '/content/drive/MyDrive/AI - Google Drive/GLM-5.2/benchmarks'
os.makedirs(BENCHMARK_OUTPUT, exist_ok=True)

# Run multi-trial benchmark
!python scripts/benchmark.py \
  --local-dir "{LOCAL_MODEL_DIR}" \
  --drive-dir "{DRIVE_MODEL_DIR}" \
  --output-dir "{BENCHMARK_OUTPUT}" \
  --repetitions 3

## Step 7: Launch OpenAI-Compatible API Gateway

In [ ]:
os.environ['COLI_HOST'] = '127.0.0.1'
os.environ['COLI_PORT'] = '8000'
os.environ['COLI_API_KEY'] = 'coli_sk_live_val_key'

# Start API server in background
import subprocess
api_proc = subprocess.Popen(['python', 'api/app.py'])

# Probe health endpoint
!python scripts/health_check.py --api-key 'coli_sk_live_val_key'